# Riken — Treino YOLOv11 Instance Segmentation
Dataset: Roboflow `walters-workspace-0ukfc/riken`

In [ ]:
# 1. Instalar dependências
!pip install roboflow ultralytics -q

In [ ]:
# 2. Verificar GPU
!nvidia-smi

In [ ]:
# 3. Baixar dataset do Roboflow
from roboflow import Roboflow

API_KEY = "LbEi5DERR6ps6EhudusB"  # sua Private API Key

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("walters-workspace-0ukfc").project("riken")
dataset = project.version(2).download("yolov11")

In [ ]:
# 4. Treinar YOLOv11
from ultralytics import YOLO

model = YOLO("yolo11n-seg.pt")  # nano-seg: rápido no Colab gratuito
                                 # troque por yolo11s-seg.pt ou yolo11m-seg.pt se tiver GPU melhor

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,       # early stop se não melhorar em 20 épocas
    device=0,          # GPU
    project="riken",
    name="v2_seg",
)

In [ ]:
# 5. Avaliar o modelo
metrics = model.val()
print("mAP50:", metrics.seg.map50)
print("mAP50-95:", metrics.seg.map)

In [ ]:
# 6. Testar inferência em uma imagem
import glob
from IPython.display import Image, display

# pega a primeira imagem do dataset de validação
val_images = glob.glob(f"{dataset.location}/valid/images/*")[:1]

results = model.predict(val_images[0], conf=0.4, save=True, project="riken", name="preview")
display(Image(filename=results[0].save_dir / results[0].path))

In [ ]:
# 7. Exportar modelo
model.export(format="onnx")   # para deploy

# Caminho do melhor modelo treinado
import os
best = "riken/v2_seg/weights/best.pt"
print(f"Modelo salvo em: {best}")
print(f"Tamanho: {os.path.getsize(best) / 1e6:.1f} MB")

In [ ]:
# 8. (Opcional) Fazer download do modelo para sua máquina
from google.colab import files
files.download("riken/v2_seg/weights/best.pt")